In [ ]:
import OpenEXR
import Imath
import numpy as np
from scipy.io import loadmat, savemat
import os
from natsort import natsorted
import h5py
from scipy import interpolate

lam = np.arange(420e-9, 730e-9, 10e-9)

def interpolate_HS_Cube(new_channels_nm, hs_cube, hs_bands):
    # Throw an error if we try to extrapolate
    if (min(new_channels_nm) < min(hs_bands) - 1) or (
        max(new_channels_nm) > max(hs_bands) + 1
    ):
        raise ValueError(
            f"In generator, extrapoaltion of the ARAD dataset outside of measurement data is not allowed: {min(hs_bands)}-{max(hs_bands)}"
        )

    interpfun = interpolate.interp1d(
        hs_bands,
        hs_cube,
        axis=-1,
        kind="linear",
        assume_sorted=True,
        fill_value="extrapolate",
        bounds_error=False,
    )
    resampled = interpfun(new_channels_nm)

    return resampled


In [28]:
# Repackage the raw dataset and save (float32) for easier usage (original are large)
datfold = "/home/deanhazineh/ssd2tb/KAIST_Dataset/raw_data/"
saveto = "./datasets/KAIST_repackaged/"

for fname in os.listdir(datfold):
    savename = fname[:-16]

    exr_file = OpenEXR.InputFile(os.path.join(datfold, fname))
    header = exr_file.header()
    dw = header['dataWindow']
    size = (dw.max.x - dw.min.x + 1, dw.max.y - dw.min.y + 1)
    channels = header['channels'].keys()
    img = np.zeros((size[1], size[0], len(channels)), dtype=np.float32)
    for i, channel in enumerate(channels):
        img[:, :, i] = np.frombuffer(exr_file.channel(channel, Imath.PixelType(Imath.PixelType.FLOAT)), dtype=np.float32).reshape(size[1], size[0])
    img = img[:,:,3:].astype(np.float32)

    savemat(saveto + savename + ".mat", {'hsi': img})

In [2]:
# For CASSI Renderings on the TSA Benchmark Challenge, we want to resave the cubes with 256x256 chunking, float16, and range [450 to 650] nm
# In the dataloader, will load these datacubes, grab random 256x256 chunks, render the measurements, and then train on sub-patches matching the challenge
datpath = f"./datasets/KAIST_repackaged/"
savepath = f"./datasets/CASSI_Dataset_450_650/" 
fnames = natsorted(os.listdir(datpath))
interp_ch = np.linspace(450, 650, 28)
ch = np.arange(420, 730, 10)

for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"]
    hsi = hsi / hsi.max()
    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float16)
    hdf5_file_path = savepath+f'KAIST_{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(256, 256, 28), dtype='float16')


In [2]:
# Optionally, create HSI data in range 420 to 700 at 10 nm for our multi-dataset used to train unconditional hyperspectral models
# Enable patch loading at 64x64
datpath = f"./datasets/KAIST_repackaged/"
savepath = f"./datasets/HSI_multiset_420_700/" 
fnames = natsorted(os.listdir(datpath))
interp_ch = np.linspace(450, 650, 28)
ch = np.arange(420, 730, 10)

for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"]
    hsi = hsi / hsi.max()
    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float16)
    hdf5_file_path = savepath+f'KAIST_{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(256, 256, 28), dtype='float16')
